In [ ]:
# ==============================================================================
# INSTALACIÓN DE DEPENDENCIAS
# ==============================================================================
#
# Se fijan versiones compatibles para garantizar la interoperabilidad entre
# ClimaX y sus dependencias. El reinicio del entorno asegura que las nuevas
# bibliotecas sean cargadas antes de continuar con el experimento.
# ==============================================================================

print("[INFO] Instalando dependencias compatibles...")

!pip -q uninstall -y numpy pandas timm climax

!pip -q install numpy==1.26.4
!pip -q install pandas==2.2.2
!pip -q install timm==0.6.13
!pip -q install git+https://github.com/microsoft/ClimaX.git

print("[OK] Instalación terminada.")

import os

os.kill(os.getpid(), 9)

[INFO] Instalando dependencias compatibles...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 109.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bokeh 3.8.2 requires pandas>=1.2, which is not installed.
geopandas 1.1.4 requires pandas>=2.0.0, which is not installed.
inequality 1.1.2 requires pandas>=2.1, which is not installed.
xarray 2025.12.0 requires pandas>=2.2, which is not installed.
dask-cuda 26.2.0 requires pandas>=1.3, which is not installed.
access 1.1.10.post3 requires pandas>=2.1.0, which is not installed.
yfinance 0.2.66 requires pandas>=1.3.0, which is not installed.
gradio 6.20.0 requires pandas<4.0,>=1.0, which is not installed.
tobler 0.14.0 requires pandas>=2.2, which is not installed.
statsmodels 0.14.6 requires pandas!=2.1.0,>=1.4, wh

In [ ]:
# ==============================================================================
# COMPATIBILIDAD CON NUMPY
# ==============================================================================
#
# ClimaX depende de alias eliminados en versiones recientes de NumPy. Se
# restauran temporalmente para mantener compatibilidad sin modificar el código
# fuente de las dependencias.
# ==============================================================================

import numpy as np

np.float = float
np.int = int
np.bool = bool
np.object = object
np.complex = complex

print("[OK] Compatibilidad NumPy aplicada.")

✅ Compatibilidad NumPy aplicada.


In [ ]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS Y DEPENDENCIAS
# ==============================================================================
#
# Las dependencias se centralizan para mejorar la reproducibilidad y evitar
# importaciones redundantes durante la ejecución del notebook.
# ==============================================================================

import os

import torch
import torch.nn as nn
import torch.nn.functional as F

from climax.arch import ClimaX
from google.colab import drive
from tqdm import tqdm

print("[INFO] Montando Google Drive...")
drive.mount("/content/drive")

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DEL EXPERIMENTO
# ==============================================================================
#
# La parametrización del horizonte temporal permite reutilizar el mismo flujo
# para los distintos escenarios de predicción sin modificar el resto del código.
# ==============================================================================

# ==============================================================================
# CONFIGURACIÓN DEL PROYECTO (Rutas Parametrizables)
# ==============================================================================
# Si cambias de entorno (Local vs Colab) o renombras la carpeta,
# solo necesitas modificar esta única variable.

BASE_PATH = "/content/drive/My Drive/Experimentos/ClimaXGanador"

SCENARIO = "largo_plazo"

SCENARIOS = {
    "corto_plazo": {
        "seq_len": 24,
        "pred_len": 3,
        "stride": 3,
    },
    "medio_plazo": {
        "seq_len": 96,
        "pred_len": 48,
        "stride": 12,
    },
    "largo_plazo": {
        "seq_len": 168,
        "pred_len": 72,
        "stride": 24,
    },
}

config = SCENARIOS[SCENARIO]

SEQ_LEN = config["seq_len"]
PRED_LEN = config["pred_len"]
STRIDE = config["stride"]

TENSORS_DIR = os.path.join(
    BASE_PATH,
    "data",
    "climax_multiscenario_tensors",
    SCENARIO,
)

MODEL_PATH = os.path.join(
    BASE_PATH,
    "models",
    "climax_finetuned",
    SCENARIO,
    "climax_best.pt",
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[INFO] Dispositivo : {device}")
print(f"[INFO] Escenario   : {SCENARIO}")
print(f"[INFO] Tensores    : {TENSORS_DIR}")
print(f"[INFO] Modelo      : {MODEL_PATH}")

print("-" * 60)
print(f"Lookback : {SEQ_LEN} horas")
print(f"Forecast : {PRED_LEN} horas")
print(f"Stride   : {STRIDE} horas")
print("-" * 60)

🔗 Montando Google Drive...
Mounted at /content/drive
Dispositivo : cuda
📦 Escenario activo: largo_plazo
📁 Tensor path: /content/drive/My Drive/Experimentos/final/data/climax_multiscenario_tensors/largo_plazo
🤖 Modelo: /content/drive/My Drive/Experimentos/final/models/climax_finetuned/largo_plazo/climax_best.pt
🌦️ RAF CONFIGURACIÓN ACTIVA
Escenario : largo_plazo
Lookback  : 168 horas
Forecast  : 72 horas
Stride    : 24 horas


**Para base de conocimiento juntamos train y validation**

In [ ]:
# ==============================================================================
# CARGA DEL CONJUNTO DE DATOS
# ==============================================================================
#
# Para la construcción de la base RAF se utilizan conjuntamente los conjuntos
# de entrenamiento y validación, maximizando la cobertura del espacio de
# representaciones disponibles.
# ==============================================================================

train_path = os.path.join(
    TENSORS_DIR,
    "climax_train.pt",
)

val_path = os.path.join(
    TENSORS_DIR,
    "climax_val.pt",
)

train_data = torch.load(train_path)
val_data = torch.load(val_path)

X_train = train_data["X"]
y_train = train_data["Y"]

X_val = val_data["X"]
y_val = val_data["Y"]

print(f"[INFO] Train shape : {X_train.shape}")
print(f"[INFO] Val shape   : {X_val.shape}")

X_all = torch.cat(
    [X_train, X_val],
    dim=0,
)

print(f"[INFO] Total ventanas RAF : {X_all.shape[0]}")

Train shape: torch.Size([5107, 168, 7, 4, 4])
Val shape: torch.Size([730, 168, 7, 4, 4])
📦 Total ventanas RAF: 5837


In [ ]:
# ==============================================================================
# MODELO CLIMAX
# ==============================================================================
#
# Se reutiliza el encoder preentrenado de ClimaX para construir una
# representación latente de cada ventana temporal. Un adaptador lineal
# transforma la dimensión temporal y un decodificador reconstruye el campo
# espacial asociado al horizonte de predicción.
# ==============================================================================
# si da error de importación, descomentar la siguiente línea y comentar
# la línea de importación de ClimaX
# from climax.arch import ClimaX


class ClimaXFineTuningWrapper(nn.Module):
    """Adaptador de fine-tuning para predicción multitemporal."""

    def __init__(
        self,
        default_vars,
        img_size=(4, 4),
        patch_size=2,
        seq_len=SEQ_LEN,
        pred_len=PRED_LEN,
    ):
        super().__init__()

        self.default_vars = default_vars
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.img_size = img_size

        self.backbone = ClimaX(
            default_vars=default_vars,
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=1024,
            depth=8,
            num_heads=16,
        )

        self.temporal_adapter = nn.Linear(
            seq_len,
            pred_len,
        )

        self.decoder = nn.Sequential(
            nn.Linear(1024, 256),
            nn.GELU(),
            nn.Linear(256, 1),
        )

    def forward(self, x):

        batch_size = x.shape[0]

        # ClimaX procesa una única imagen por instancia.

        x = x.reshape(
            batch_size * self.seq_len,
            x.shape[2],
            x.shape[3],
            x.shape[4],
        )

        lead_times = torch.zeros(
            batch_size * self.seq_len,
            device=x.device,
            dtype=torch.float32,
        )

        features = self.backbone.forward_encoder(
            x,
            lead_times,
            self.default_vars,
        )

        features = features.view(
            batch_size,
            self.seq_len,
            features.shape[1],
            features.shape[2],
        )

        # La adaptación temporal se realiza sobre las representaciones
        # espaciales aprendidas por el encoder.

        features = features.permute(
            0,
            2,
            3,
            1,
        )

        features = self.temporal_adapter(features)

        features = features.permute(
            0,
            3,
            1,
            2,
        )

        output = self.decoder(features)

        output = output.squeeze(-1)

        output = output.view(
            batch_size,
            self.pred_len,
            2,
            2,
        )

        # Se reconstruye la resolución espacial original mediante
        # interpolación bilineal.

        output = F.interpolate(
            output.reshape(
                batch_size * self.pred_len,
                1,
                2,
                2,
            ),
            size=self.img_size,
            mode="bilinear",
            align_corners=False,
        )

        output = output.view(
            batch_size,
            self.pred_len,
            1,
            self.img_size[0],
            self.img_size[1],
        )

        return output

In [ ]:
# ==============================================================================
# CARGA DEL MODELO
# ==============================================================================
#
# Se restauran los pesos obtenidos durante el proceso de fine-tuning para
# reutilizar el encoder como generador de representaciones latentes.
# ==============================================================================
variables_meteorologicas = [
    "temperaturaMedia",
    "humedadRelativa",
    "presionBarometrica",
    "precipitacion",
    "radiacionSolar",
    "viento_u",
    "viento_v",
]

model = ClimaXFineTuningWrapper(
    default_vars=variables_meteorologicas,
    seq_len=SEQ_LEN,
    pred_len=PRED_LEN,
)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device,
    )
)

model.to(device)
model.eval()

print("[OK] Modelo cargado correctamente.")

✅ Modelo listo


In [ ]:
# ==============================================================================
# EXTRACCIÓN DE EMBEDDINGS
# ==============================================================================
#
# Los embeddings se obtienen directamente del encoder preentrenado mediante
# agregación temporal y espacial, preservando la representación latente
# aprendida durante el fine-tuning.
# ==============================================================================


def get_embedding(x):

    with torch.no_grad():

        x = x.to(device)

        batch_size, seq_len, channels, height, width = x.shape

        x = x.view(
            batch_size * seq_len,
            channels,
            height,
            width,
        )

        lead_times = torch.zeros(
            batch_size * seq_len,
            device=device,
            dtype=torch.float32,
        )

        features = model.backbone.forward_encoder(
            x,
            lead_times,
            model.default_vars,
        )

        features = features.view(
            batch_size,
            seq_len,
            features.shape[1],
            features.shape[2],
        )

        embeddings = features.mean(
            dim=(1, 2),
        )

    return embeddings.cpu()

In [ ]:
# ==============================================================================
# CONSTRUCCIÓN DE LA BASE RAF
# ==============================================================================
#
# Los embeddings se generan por lotes para controlar el consumo de memoria GPU.
# Posteriormente se normalizan para facilitar el cálculo de similitud mediante
# producto punto o similitud coseno.
# ==============================================================================
BATCH = 32

embeddings = []

print("[INFO] Construyendo base RAF...")

for i in tqdm(
    range(0, len(X_all), BATCH),
):

    x_batch = X_all[i : i + BATCH]

    embedding = get_embedding(x_batch)

    embeddings.append(embedding)

embeddings = torch.cat(
    embeddings,
    dim=0,
)

embeddings = embeddings / (
    embeddings.norm(
        dim=1,
        keepdim=True,
    )
    + 1e-8
)

print(f"[OK] Base RAF creada: {embeddings.shape}")

🔄 Construyendo base RAF optimizada...
✅ Base RAF lista: torch.Size([5837, 1024])


In [ ]:
MODEL_PATH

'/content/drive/My Drive/Experimentos/final/models/climax_finetuned/largo_plazo/climax_best.pt'

In [ ]:
# ==============================================================================
# EXPORTACIÓN DE EMBEDDINGS
# ==============================================================================
#
# Los embeddings normalizados se almacenan en formato PyTorch para conservar
# compatibilidad con cargas posteriores en CPU o GPU. La persistencia separada
# permite reutilizar la base RAF sin repetir la extracción del encoder.
# ==============================================================================

RAF_PATH = os.path.join(
    BASE_PATH,
    "data",
    "climax_multiscenario_tensors",
    SCENARIO,
)

EMB_PATH = os.path.join(
    RAF_PATH,
    "embeddings.pt",
)

torch.save(
    embeddings.cpu(),
    EMB_PATH,
)

print(f"[OK] Embeddings guardados en: {EMB_PATH}")

💾 Embeddings guardados en: /content/drive/My Drive/Experimentos/final/data/climax_multiscenario_tensors/largo_plazo/embeddings.pt


In [ ]:
# ==============================================================================
# EXPORTACIÓN DE METADATOS RAF
# ==============================================================================
#
# La metadata asociada permite reconstruir la configuración temporal utilizada
# durante la generación de la base de recuperación y mantener trazabilidad del
# experimento.
# ==============================================================================

IDX_PATH = os.path.join(
    RAF_PATH,
    "indices.pt",
)

raf_metadata = {
    "scenario": SCENARIO,
    "seq_len": SEQ_LEN,
    "pred_len": PRED_LEN,
    "stride": STRIDE,
    "num_samples": len(embeddings),
}

torch.save(
    raf_metadata,
    IDX_PATH,
)

print(f"[OK] Metadata guardada en: {IDX_PATH}")

📦 Metadata guardada en:  /content/drive/My Drive/Experimentos/final/data/climax_multiscenario_tensors/largo_plazo/indices.pt


In [ ]:
# ==============================================================================
# EXPORTACIÓN EN FORMATO NUMPY
# ==============================================================================
#
# Se genera una copia compatible con herramientas externas de análisis y
# búsqueda vectorial. El formato NumPy facilita la integración con librerías
# especializadas en indexación y similitud.
# ==============================================================================

NPY_PATH = EMB_PATH.replace(
    ".pt",
    ".npy",
)

np.save(
    NPY_PATH,
    embeddings.cpu().numpy(),
)

print(f"[OK] Archivo NumPy generado en: {NPY_PATH}")